# LangGraph Environment Setup and Telemetry Infrastructure
**Production Reference Implementation (Senior AI Engineer Standard)**

## Architectural Scope
This notebook establishes and audits the local execution environment for **LangGraph** applications.
It validates:
1. Runtime environment isolation (Python 3.12+ execution boundary and prefix path).
2. Package dependency availability (`langgraph`, `langchain-core`, `langchain-community`, `ipykernel`).
3. Secrets management and `.env` loading with credential safety checks.
4. Telemetry and observability configuration (LangSmith tracing hooks).
5. Deterministic graph smoke-test executing an environment health-check state machine.
6. Programmatic validation assertions ensuring deterministic runtime readiness.


In [1]:
import sys
import os
from typing import Dict, Any, List, TypedDict, Optional
import dotenv

print("Python Executable Path:", sys.executable)
print("Python Version Info:", sys.version.replace("\n", " "))
is_venv = hasattr(sys, 'real_prefix') or (hasattr(sys, 'base_prefix') and sys.base_prefix != sys.prefix)
print("Isolated Virtual Environment Active:", is_venv)


Python Executable Path: D:\Code\AI\rag-mastery\.venv\Scripts\python.exe
Python Version Info: 3.12.12 (main, Feb 12 2026, 00:40:26) [MSC v.1944 64 bit (AMD64)]
Isolated Virtual Environment Active: True


In [2]:
def audit_package_dependencies() -> Dict[str, str]:
    """Audits presence and versions of foundational LangGraph packages."""
    packages = [
        "langgraph",
        "langchain_core",
        "langchain_community",
        "langchain_groq",
        "ipykernel",
        "dotenv"
    ]
    status = {}
    for pkg in packages:
        try:
            mod = __import__(pkg)
            version = getattr(mod, "__version__", "installed (no version attr)")
            status[pkg] = version
        except ImportError as e:
            status[pkg] = f"MISSING ({e})"
    return status

dependency_report = audit_package_dependencies()
for pkg, ver in dependency_report.items():
    print(f"Dependency [{pkg}]: {ver}")


C:\Users\aliashraf\AppData\Local\Temp\ipykernel_28700\1011516949.py:14: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  mod = __import__(pkg)


Dependency [langgraph]: installed (no version attr)
Dependency [langchain_core]: 1.6.6
Dependency [langchain_community]: 0.4.2
Dependency [langchain_groq]: 1.1.3
Dependency [ipykernel]: 7.4.0
Dependency [dotenv]: installed (no version attr)


## 1. Secrets Management and Observability Hooks
We load environment configurations from `.env` and verify API key presence without leaking secrets.
We also configure LangSmith telemetry environment variables for automated graph tracing.


In [3]:
dotenv.load_dotenv()

# Audit credentials safely
groq_key: Optional[str] = os.getenv("GROQ_API_KEY")
openai_key: Optional[str] = os.getenv("OPENAI_API_KEY")
langsmith_key: Optional[str] = os.getenv("LANGSMITH_API_KEY") or os.getenv("LANGCHAIN_API_KEY")

print("Credentials Status:")
print("  GROQ_API_KEY Available:", bool(groq_key))
print("  OPENAI_API_KEY Available:", bool(openai_key))
print("  LANGSMITH_API_KEY Available:", bool(langsmith_key))

# Configure Telemetry settings
tracing_enabled = os.getenv("LANGCHAIN_TRACING_V2", "false").lower() == "true"
project_name = os.getenv("LANGCHAIN_PROJECT", "rag-mastery-langgraph")

print(f"\nTelemetry Configuration:")
print(f"  LangSmith Tracing Active: {tracing_enabled}")
print(f"  Active Project Namespace: {project_name}")


Credentials Status:
  GROQ_API_KEY Available: True
  OPENAI_API_KEY Available: False
  LANGSMITH_API_KEY Available: False

Telemetry Configuration:
  LangSmith Tracing Active: False
  Active Project Namespace: rag-mastery-langgraph


## 2. Deterministic Graph Smoke Test
We define a minimal diagnostic state graph to verify that `langgraph` compiles,
executes nodes, updates state, and halts cleanly in this environment.


In [4]:
from langgraph.graph import StateGraph, START, END

class EnvironmentAuditState(TypedDict):
    """Schema tracking environment diagnostics during the smoke test."""
    runtime_ok: bool
    dependencies_ok: bool
    secrets_ok: bool
    audit_log: List[str]

def audit_runtime_node(state: EnvironmentAuditState) -> Dict[str, Any]:
    """Validates python version and environment boundaries."""
    version_valid = sys.version_info >= (3, 10)
    log_entry = f"Python {sys.version_info.major}.{sys.version_info.minor} verified."
    return {
        "runtime_ok": version_valid,
        "audit_log": state["audit_log"] + [log_entry]
    }

def audit_packages_node(state: EnvironmentAuditState) -> Dict[str, Any]:
    """Validates that all essential packages are accessible."""
    all_present = all("MISSING" not in ver for ver in dependency_report.values())
    log_entry = "Core LangGraph dependencies verified." if all_present else "Dependency missing."
    return {
        "dependencies_ok": all_present,
        "audit_log": state["audit_log"] + [log_entry]
    }

def audit_secrets_node(state: EnvironmentAuditState) -> Dict[str, Any]:
    """Validates that at least one primary inference provider credential exists."""
    has_provider = bool(groq_key or openai_key)
    log_entry = "Primary inference provider credential verified." if has_provider else "Warning: No inference keys."
    return {
        "secrets_ok": has_provider,
        "audit_log": state["audit_log"] + [log_entry]
    }

print("Diagnostic nodes defined: audit_runtime, audit_packages, audit_secrets.")


Diagnostic nodes defined: audit_runtime, audit_packages, audit_secrets.


In [5]:
# Build diagnostic graph
workflow = StateGraph(EnvironmentAuditState)

workflow.add_node("runtime_check", audit_runtime_node)
workflow.add_node("packages_check", audit_packages_node)
workflow.add_node("secrets_check", audit_secrets_node)

workflow.add_edge(START, "runtime_check")
workflow.add_edge("runtime_check", "packages_check")
workflow.add_edge("packages_check", "secrets_check")
workflow.add_edge("secrets_check", END)

smoke_app = workflow.compile()

# Execute smoke test
initial_state = {
    "runtime_ok": False,
    "dependencies_ok": False,
    "secrets_ok": False,
    "audit_log": []
}

final_audit = smoke_app.invoke(initial_state)

print("Diagnostic Smoke Test Execution Results:")
print("  Runtime Status:", final_audit["runtime_ok"])
print("  Dependencies Status:", final_audit["dependencies_ok"])
print("  Secrets Status:", final_audit["secrets_ok"])
print("  Audit Log History:")
for entry in final_audit["audit_log"]:
    print(f"    - {entry}")


Diagnostic Smoke Test Execution Results:
  Runtime Status: True
  Dependencies Status: True
  Secrets Status: True
  Audit Log History:
    - Python 3.12 verified.
    - Core LangGraph dependencies verified.
    - Primary inference provider credential verified.


## 3. Programmatic Readiness Assertions
Defensive validation confirming environment compliance with production standards.


In [6]:
# 1. Verify runtime compliance
assert sys.version_info >= (3, 10), f"Assertion Failed: Python 3.10+ required, got {sys.version_info}."

# 2. Verify dependency integrity
assert "MISSING" not in dependency_report["langgraph"], "Assertion Failed: langgraph package is missing."
assert "MISSING" not in dependency_report["langchain_core"], "Assertion Failed: langchain_core package is missing."
assert "MISSING" not in dependency_report["ipykernel"], "Assertion Failed: ipykernel package is missing."

# 3. Verify smoke test outputs
assert final_audit["runtime_ok"] is True, "Assertion Failed: Runtime audit failed."
assert final_audit["dependencies_ok"] is True, "Assertion Failed: Dependency audit failed."
assert len(final_audit["audit_log"]) == 3, "Assertion Failed: Expected 3 audit log entries."

print("All environment readiness assertions passed successfully.")


All environment readiness assertions passed successfully.
